In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print(f"Dataset shape: {df.shape}")
df.head()


Saving CVD_cleaned.csv to CVD_cleaned.csv
Dataset shape: (308854, 19)


,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,Poor,Within the past 2 years,No,No,No,No,No,No,Yes,Female,70-74,150,32.66,14.54,Yes,0,30,16,12
1,Very Good,Within the past year,No,Yes,No,No,No,Yes,No,Female,70-74,165,77.11,28.29,No,0,30,0,4
2,Very Good,Within the past year,Yes,No,No,No,No,Yes,No,Female,60-64,163,88.45,33.47,No,4,12,3,16
3,Poor,Within the past year,Yes,Yes,No,No,No,Yes,No,Male,75-79,180,93.44,28.73,No,0,30,30,8
4,Good,Within the past year,No,No,No,No,No,No,No,Male,80+,191,88.45,24.37,Yes,0,8,4,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

target_col = df.columns[-1]
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if y.nunique() < 20 else None
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


Train shape: (247083, 18) Test shape: (61771, 18)


In [3]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

models = {
    'SVM': Pipeline([('preprocess', preprocessor),
                     ('clf', SVC(kernel='linear', random_state=42))]),
    'RandomForest': Pipeline([('preprocess', preprocessor),
                              ('clf', RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42))]),
    'LogisticRegression': Pipeline([('preprocess', preprocessor),
                                   ('clf', LogisticRegression(max_iter=300, solver='liblinear', random_state=42))])
}

print("Models initialized:", list(models.keys()))


Models initialized: ['SVM', 'RandomForest', 'LogisticRegression']


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

results = {}

for name, model in models.items():
    print(f"\nTraining and evaluating {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"{name} Accuracy: {acc:.4f}")
    print("Classification Report:\n", classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    results[name] = acc

# Plot accuracies
plt.bar(results.keys(), results.values(), color=['blue', 'green', 'orange'])
plt.ylim(0, 1)
plt.title("Model Accuracy Comparison")
plt.show()



Training and evaluating SVM...


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_distributions = {
    'SVM': {'clf__C': [0.1, 1, 10], 'clf__kernel': ['linear', 'rbf']},
    'RandomForest': {'clf__n_estimators': [30, 50], 'clf__max_depth': [5, 10, None]},
    'LogisticRegression': {'clf__C': [0.01, 0.1, 1], 'clf__solver': ['liblinear']}
}

best_models = {}
tuned_scores = {}

for name, model in models.items():
    print(f"\nTuning hyperparameters for {name}...")
    rs = RandomizedSearchCV(
        model,
        param_distributions=param_distributions[name],
        n_iter=3,
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
    rs.fit(X_train, y_train)
    best_models[name] = rs.best_estimator_
    tuned_acc = accuracy_score(y_test, rs.best_estimator_.predict(X_test))
    tuned_scores[name] = tuned_acc
    print(f"Best params for {name}: {rs.best_params_}")
    print(f"Tuned accuracy for {name}: {tuned_acc:.4f}")

plt.bar(tuned_scores.keys(), tuned_scores.values(), color=['blue', 'green', 'orange'])
plt.ylim(0, 1)
plt.title("Tuned Model Accuracy Comparison")
plt.show()

best_model_name = max(tuned_scores, key=tuned_scores.get)
best_model = best_models[best_model_name]

sample_idx = 0
sample = X_test.iloc[[sample_idx]]
true_label = y_test.iloc[sample_idx]
pred_label = best_model.predict(sample)[0]

print(f"Sample input:\n{sample}")
print(f"True label: {true_label}")
print(f"Predicted by {best_model_name}: {pred_label}")
